# Benchmark first pass on all annotated cells

In [ ]:
import os
import sys
import glob
from pyhere import here
from pathlib import Path

import anndata
import numpy as np
import pandas as pd
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import seaborn as sns
import torch
import skmisc
import subprocess
from joblib import parallel_backend
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
import warnings

import zstandard as zstd
import hdf5plugin

import pickle

# My modules / functions
sys.path.append(str(here('scripts/misc')))  
import my_anndata as ma

In [ ]:
# Saving
base_dir = str(here('data/integrate/first_pass/'))
file_dir = os.path.join(base_dir, 'files') 
plot_dir = os.path.join(base_dir, 'plot') 
tmp_dir = os.path.join(base_dir, 'tmp') 
emb_dir = os.path.join(base_dir, 'embedding') 
objects_dir = os.path.join(base_dir, 'objects') 
model_dir = os.path.join(base_dir, 'models')

In [ ]:
# hvgs 
hvgs = [500, 1000, 2000]

technical_seperate = ['cell_nuclei', 'ic_id_study_overall', 'library_prep', 'ic_id_donor_overall']

celltype = 'study_cell_annotation_harmonized' 

## Load Full adata object

In [ ]:
# Load reference adata object
adata_full =sc.read_h5ad(here('data/anndata/AB_combined.h5ad'))

In [ ]:
adata_full.obs['technical'] = (
    adata_full.obs['library_prep'].astype(str) + "_" + adata_full.obs['ic_id_study_overall'].astype(str) + "_" + adata_full.obs['cell_nuclei'].astype(str)
)

adata_full.obs['technical_donor'] = (
    adata_full.obs['technical'].astype(str) + "_" + adata_full.obs['ic_id_donor_overall'].astype(str)) 

## Add Latent space to AnnData object

In [ ]:
embd_list = glob.glob(os.path.join(base_dir, 'embedding/*'))

In [ ]:
embd_list_filtered = [f for f in embd_list if "donor" in f or "pca" in f]

In [ ]:
embd_list_filtered

In [ ]:
for embedding_path in embd_list_filtered:
    # Get obsm key
    base = os.path.basename(embedding_path)
    prefix = "_".join(base.split("_")[:2])

    # Add embedding
    adata_full = ma.add_embedding(ad=adata_full, embed_path=embedding_path, obsm_key=prefix)

## Get embedding names

In [5]:
embedding_keys = [x for x in adata_full.obsm.keys()]

In [9]:
embedding_keys = [x for x in adata_full.obsm.keys() if not x.endswith('_umap')]

## Subset data to only contain annotated cells

In [11]:
celltype = "study_cell_annotation_harmonized"

n_before = adata_full.n_obs
adata_sub = adata_full[~adata_full.obs[celltype].isin(["unknown", "excluded"])].copy()
adata_sub.obs[celltype] = adata_sub.obs[celltype].astype("category")
adata_sub.obs[celltype] = adata_sub.obs[celltype].cat.remove_unused_categories()

n_after = adata_sub.n_obs
print(f"Filtered {n_before - n_after} cells (kept {n_after})")

Filtered 323202 cells (kept 266000)


## Benchmark for subset of cells

In [12]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        category=FutureWarning,
        message="Argument `use_highly_variable` is deprecated.*"
    )
    
    results = {}
    results_norm = {}
    
    for technical in technical_seperate:

        print(f'Running {technical}')
        
        bm = Benchmarker(
            adata_sub,
            batch_key=technical,
            label_key=celltype,
            bio_conservation_metrics=BioConservation(),
            batch_correction_metrics=BatchCorrection(kbet_per_label = False),
            embedding_obsm_keys=embedding_keys,
            n_jobs=100,
        )
    
        bm.benchmark()

        # save results
        results[technical] = bm.get_results(min_max_scale=False)
        results_norm[technical] = bm.get_results(min_max_scale=True)

        bm.get_results(min_max_scale=False).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_results_{technical}.csv'), index=True)
        bm.get_results(min_max_scale=True).to_csv(os.path.join(file_dir, f'benchmark_celltype_sub_norm_results_{technical}.csv'), index=True)
    
    
        print(f"[OK] Finished benchmark for {technical}")

Running cell_nuclei


Metrics:  60%|██████    | 6/10 [02:06<00:53, 13.48s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   8%|▊         | 1/12 [02:07<23:27, 127.99s/it]ch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [02:00<00:51, 12.89s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:  17%|█▋        | 2

[OK] Finished benchmark for cell_nuclei
Running ic_id_study_overall


Metrics:  60%|██████    | 6/10 [02:17<01:06, 16.74s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   8%|▊         | 1/12 [02:18<25:25, 138.65s/it]ch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [02:14<01:03, 15.80s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:  17%|█▋        | 2

[OK] Finished benchmark for ic_id_study_overall
Running library_prep


Metrics:  60%|██████    | 6/10 [02:15<01:04, 16.24s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   8%|▊         | 1/12 [02:16<25:03, 136.73s/it]ch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [02:13<01:03, 15.76s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:  17%|█▋        | 2

[OK] Finished benchmark for library_prep
Running ic_id_donor_overall


Metrics:  60%|██████    | 6/10 [02:18<01:06, 16.71s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:   8%|▊         | 1/12 [02:22<26:08, 142.55s/it]ch correction: pcr_comparison]
                                                                                         
Metrics:  60%|██████    | 6/10 [02:13<01:02, 15.74s/it, Batch correction: graph_connectivity]/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = pd.value_counts(comps)

Embeddings:  17%|█▋        | 2

[OK] Finished benchmark for ic_id_donor_overall


/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:299: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Aggregate score' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[_METRIC_TYPE, per_class_score.columns] = _AGGREGATE_SCORE
/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:299: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Aggregate score' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[_METRIC_TYPE, per_class_score.columns] = _AGGREGATE_SCORE
/work/islet_cartography_scrna/scrna_cartography_py_analysis/lib/python3.13/site-packages/scib_metrics/benchmark/_core.py:299: FutureWarning: Set

### Load results table and combine into one dataframe and save

In [12]:
# Get path to results (not normalised)
result_path = glob.glob(os.path.join(file_dir, "benchmark*.csv"))
result_path = [x for x in result_path if "norm" not in x]

df_list = []
for res in result_path:
    # Get base name
    base_name = Path(res).stem.replace("benchmark_celltype_sub_results_", "")
    # Load file
    df_tmp = pd.read_csv(res, index_col=0)
    # Transpose, add variable name and pivot longer
    df_tmp = df_tmp.transpose()
    # Make numeric
    cols = ['pca_1000', 'pca_2000', 'pca_500', 'scpoli_1000', 'scpoli_2000',
       'scpoli_500', 'scvi_1000', 'scvi_2000', 'scvi_500', 'sysvi_1000',
       'sysvi_2000', 'sysvi_500']
    df_tmp[cols] = df_tmp[cols].apply(pd.to_numeric, errors='coerce')
    
    # Drop cluster dependet metrics and precalculated aggregate scores
    # As we are not going to cluster based on the KNN, and as these labels are from studies that have not been corrected
    df_tmp = df_tmp.drop(['KMeans NMI', 'KMeans ARI', 'Total', 'Batch correction', 'Bio conservation'])

    # Calculate aggregate scores
    df_means = df_tmp.groupby('Metric Type').mean(numeric_only=True)
    df_means = df_means.transpose()
    df_means['Total'] = 0.5 * df_means["Batch correction"] + 0.5 * df_means["Bio conservation"]
    df_means = df_means.transpose()
    
    # Prepare for combining with df_tmp
    df_means.index.name = 'Embedding'  # to align with df_tmp index
    df_means['Metric Type'] = 'Aggregate score'

    # combine with tmp
    df_tmp = pd.concat([df_tmp, df_means])
    df_tmp["Variable"] = base_name
    df_tmp = df_tmp.rename_axis("Metric").reset_index()
    df_tmp = df_tmp.melt(
        id_vars=["Metric", "Metric Type", "Variable"],
        var_name="Latent",
        value_name="Score",
    )

    # Add to list
    df_list.append(df_tmp)

# Combine results to one df
df_res = pd.concat(df_list)

# round to 4 digits
df_res = df_res.round(2)
print(df_res)

# Save results
df_res.to_csv(os.path.join(file_dir, 'benchmark_results_per_variable.csv'))

                 Metric       Metric Type             Variable     Latent  \
0       Isolated labels  Bio conservation  ic_id_donor_overall   pca_1000   
1      Silhouette label  Bio conservation  ic_id_donor_overall   pca_1000   
2                 cLISI  Bio conservation  ic_id_donor_overall   pca_1000   
3                  BRAS  Batch correction  ic_id_donor_overall   pca_1000   
4                 iLISI  Batch correction  ic_id_donor_overall   pca_1000   
..                  ...               ...                  ...        ...   
115  Graph connectivity  Batch correction         library_prep  sysvi_500   
116      PCR comparison  Batch correction         library_prep  sysvi_500   
117    Batch correction   Aggregate score         library_prep  sysvi_500   
118    Bio conservation   Aggregate score         library_prep  sysvi_500   
119               Total   Aggregate score         library_prep  sysvi_500   

     Score  
0     0.59  
1     0.65  
2     1.00  
3     0.63  
4     0.01

### Total score

In [14]:
# Mean of aggregated scores
df_total = df_res[
    (df_res["Metric"].isin(["Bio conservation", "Batch correction", "Total"]))
].groupby(['Latent','Metric']).mean(numeric_only=True).round(2)

print(df_total)
df_res.to_csv(os.path.join(file_dir,'benchmark_results_total.csv'))

                              Score
Latent      Metric                 
pca_1000    Batch correction   0.53
            Bio conservation   0.75
            Total              0.64
pca_2000    Batch correction   0.54
            Bio conservation   0.74
            Total              0.64
pca_500     Batch correction   0.55
            Bio conservation   0.75
            Total              0.65
scpoli_1000 Batch correction   0.61
            Bio conservation   0.72
            Total              0.66
scpoli_2000 Batch correction   0.61
            Bio conservation   0.72
            Total              0.66
scpoli_500  Batch correction   0.60
            Bio conservation   0.73
            Total              0.66
scvi_1000   Batch correction   0.61
            Bio conservation   0.73
            Total              0.66
scvi_2000   Batch correction   0.61
            Bio conservation   0.72
            Total              0.67
scvi_500    Batch correction   0.61
            Bio conservation

## Add find neighbors and run umap

In [ ]:
from joblib import Parallel, delayed, parallel_backend

# Function to compute neighbors for one embedding
def compute_neighbors(key):
    neig_key = f"{key}_neighbors"
    sc.pp.neighbors(
        adata_full, 
        use_rep=key, 
        key_added=neig_key
    )

# Run in parallel with threading backend
with parallel_backend("threading", n_jobs=100):  # controls number of threads
    Parallel(n_jobs=len(embedding_keys))(
        delayed(compute_neighbors)(k) for k in embedding_keys
    )


In [ ]:
# Function to compute UMAP for one neighbors graph
def compute_umap(neighbors_key):
    umap_key = neighbors_key.replace("_neighbors", "_umap")
    sc.tl.umap(
        adata_full, 
        neighbors_key=neighbors_key, 
        key_added=umap_key
    )

neighbors_keys = [f"{key}_neighbors" for key in embedding_keys]

# Run sequentially 
for nk in neighbors_keys:
    compute_umap(nk)

In [ ]:
adata_full.write_h5ad(os.path.join(objects_dir, 'adata_full.h5ad'))